# 08 — DAEs, quadratures, and sensitivities

**Objectives.** Form a semi-explicit index-1 DAE, integrate it with IDAS, and differentiate the terminal state. A semi-explicit DAE has $\dot x=f(x,z,p)$ and $0=g(x,z,p)$. IDAS needs a consistent algebraic initial guess; high-index DAEs generally require reformulation.


In [ ]:
import casadi as ca
x = ca.MX.sym('x'); z = ca.MX.sym('z'); p = ca.MX.sym('p')
dae = {'x':x, 'z':z, 'p':p, 'ode':-z, 'alg':z**2-x-p, 'quad':x**2}
F = ca.integrator('dae_step', 'idas', dae, 0.0, 0.2, {'abstol':1e-9, 'reltol':1e-9})
result = F(x0=1.0, z0=1.0, p=0.0)
print('xf', result['xf'], 'zf', result['zf'], 'integral x^2 dt', result['qf'])
assert abs(float(result['zf']**2-result['xf'])) < 1e-7


In [ ]:
p_symbol = ca.MX.sym('p_symbol')
terminal = F(x0=1.0, z0=1.0, p=p_symbol)['xf']
terminal_and_sensitivity = ca.Function('terminal_sensitivity', [p_symbol], [terminal, ca.jacobian(terminal,p_symbol)])
print(terminal_and_sensitivity(0.0))


Quadratures accumulate costs without adding artificial differential states. CasADi constructs the derivative equations needed for forward/reverse sensitivities around supported integrators.

**Failure modes:** inconsistent `z0`, singular $\partial g/\partial z$, hidden high index, and discontinuities. **Recap:** establish consistency and residual checks before requesting sensitivities.
